# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [38]:
ПРІЗВИЩЕ_ІМЯ = 'Смачило Іванна'      # напишіть тут своє прізвище та ім'я
ГРУПА        = 'ШІ-33'
НОМЕР        = 11       # ваш номер за списком групи (1–30)

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [39]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 11   ·   Смачило Іванна   ·   ШІ-33

Ваші вісім стовпців:
   1. Телефон
   2. Примітка
   3. ЛН
   4. Виписка
   5. Ліжко днів
   6. Прокальцитонін
   7. Білок1
   8. Гематокрит2

Ваша пара для аналізу зв’язку: Вік та spO2 без дот

Ваше додаткове питання:
   Стовпець «Вік» можна перетворити на вікові групи «до 40», «40–60», «понад 60». Яка шкала була до перетворення і яка стала? Що ви при цьому втратили і що здобули?


In [40]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,Телефон,Примітка,ЛН,Виписка,Ліжко днів,Прокальцитонін,Білок1,Гематокрит2
0,0958469788 дружина Ганна,NaN,ІІІ-IV,2022-03-10 00:00:00,14,0.126,70.0,45.0
1,"0671778486, 0994887663 дружина Оксана","отримував актемра, часті носові кровотечі, втрата нюху",ІІ-ІІІ,2022-03-16 00:00:00,22,NaN,61.8,57.0
2,"0958768045, чоловік Степан 0951784726",часта втрата свідомосі в стаціонарі,ІІ,2022-01-18 00:00:00,20,NaN,80.5,37.0
3,0959819331. дочка Валентина 0960845205,панкреонекроз,ІІІ-IV,2022-01-13 00:00:00,20,NaN,62.6,34.0
4,"0952294866, дочка Дарина 0678360161",NaN,ІІ-ІІІ,2022-03-11 00:00:00,27,NaN,72.0,48.0
5,0959922983 сестра Софія,NaN,ІІ-ІІІ,2021-12-26 00:00:00,9(5 у реан),NaN,73.0,36.0
6,"0998073702, дружина Наталія 0998382226","глюкозурія, кетонові тіла в сечі, ЦД, фібриляція передс",ІІ-ІІІ,2022-05-11 00:00:00,18,NaN,76.0,44.5
7,"0637258771, син Сергій 0968413301",NaN,І,2022-05-28 00:00:00,10,NaN,88.0,42.0


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [41]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Телефон         120 non-null    object 
 1   Примітка        45 non-null     object 
 2   ЛН              121 non-null    object 
 3   Виписка         120 non-null    object 
 4   Ліжко днів      120 non-null    object 
 5   Прокальцитонін  46 non-null     float64
 6   Білок1          118 non-null    float64
 7   Гематокрит2     101 non-null    float64
dtypes: float64(3), object(5)
memory usage: 7.7+ KB


In [42]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- Телефон
    непорожніх 120 з 121 | унікальних 120 | типи: str×97, int×23
    приклади: ["'0958469788 дружина Ганна'", "'0671778486, 0994887663 дружина Оксана'", "'0958768045, чоловік Степан 0951784726'", "'0959819331. дочка Валентина 0960845205'"]

--- Примітка
    непорожніх 45 з 121 | унікальних 44 | типи: str×45
    приклади: ["'отримував актемра, часті носові кровотечі, втрата нюху'", "'часта втрата свідомосі в стаціонарі'", "'панкреонекроз'", "'глюкозурія, кетонові тіла в сечі, ЦД, фібриляція передс'"]

--- ЛН
    непорожніх 121 з 121 | унікальних 6 | типи: str×121
    приклади: ["'ІІІ-IV'", "'ІІ-ІІІ'", "'ІІ'", "'І'"]

--- Виписка
    непорожніх 120 з 121 | унікальних 90 | типи: datetime×118, str×2
    приклади: ['datetime.datetime(2022, 3, 10, 0, 0)', 'datetime.datetime(2022, 3, 16, 0, 0)', 'datetime.datetime(2022, 1, 18, 0, 0)', 'datetime.datetime(2022, 1, 13, 0, 0)']

--- Ліжко днів
    непорожніх 120 з 121 | унікальних 40 | типи: int×119, str×1
    приклади: ['14', '22', '

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [43]:
ПАСПОРТ = {
    # 'назва стовпця': ('шкала', 'обґрунтування однією фразою'),
    'Телефон': ('ідентифікатор', 'Номер телефону є унікальним контактом пацієнта над яким не мають сенсу жодні арифметичні операції.'),
    'Примітка': ('номінальна', 'Довільний текстовий коментар або категорія, яку можна лише порівнювати на збіг чи рівність.'),
    'ЛН': ('порядкова', 'Категорія листка непрацездатності має природний порядок, але точні відстані між рівнями не визначені.'),
    'Виписка': ('інтервальна', 'Календарна дата виписки має умовний початок відліку, при цьому різниця між датами вимірюється в днях.'),
    'Ліжко днів': ('відношень', 'Кількість днів у стаціонарі має природний абсолютний нуль і дозволяє коректне порівняння через відношення.'),
    'Прокальцитонін': ('відношень', 'Біохімічний показник крові має природний абсолютний нуль і пропорційну шкалу вимірювання.'),
    'Білок1': ('відношень', 'Концентрація білка при першому обстеженні вимірюється в абсолютній шкалі з природним нулем.'),
    'Гематокрит2': ('абсолютна', 'Відсотковий показник крові має природний нуль і фіксовану безрозмірнісну одиницю вимірювання.'),
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,Телефон,ідентифікатор,Номер телефону є унікальним контактом пацієнта над яким ...
1,Примітка,номінальна,"Довільний текстовий коментар або категорія, яку можна ли..."
2,ЛН,порядкова,"Категорія листка непрацездатності має природний порядок,..."
3,Виписка,інтервальна,"Календарна дата виписки має умовний початок відліку, при..."
4,Ліжко днів,відношень,Кількість днів у стаціонарі має природний абсолютний нул...
5,Прокальцитонін,відношень,Біохімічний показник крові має природний абсолютний нуль...
6,Білок1,відношень,Концентрація білка при першому обстеженні вимірюється в ...
7,Гематокрит2,абсолютна,Відсотковий показник крові має природний нуль і фіксован...


In [44]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [45]:
def допустима_статистика(стовпець, шкала):
    """Повертає словник статистик, дозволених для цієї шкали."""
    з = df[стовпець].dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()

    if ш in ('ідентифікатор',):
        return р
    mod = з.mode()
    р['мода'] = mod.tolist() if not mod.empty else None
    
    if ш in ('порядкова', 'інтервальна', 'відношень', 'абсолютна'):
        try:
            р['медіана'] = float(з.гельті() if hasattr(з, 'median') else з.median())
            р['медіана'] = float(з.median())
            р['квантиль_25'] = float(з.quantile(0.25))
            р['квантиль_75'] = float(з.quantile(0.75))
        except Exception:
            pass
        
    if ш in ('інтервальна', 'відношень', 'абсолютна'):
        з_num = pd.to_numeric(з, errors='coerce').dropna()
        if len(з_num) > 0:
            р['середнє'] = float(з_num.mean())
            р['std'] = float(з_num.std())
        
    if ш in ('відношень', 'абсолютна'):
        з_num = pd.to_numeric(з, errors='coerce').dropna()
        if len(з_num) > 0 and з_num.mean() != 0:
            р['коеф_варіації'] = float(з_num.std() / з_num.mean())
            
    return р


for c in МОЇ:
    print(f'--- {c}  ({ПАСПОРТ[c][0]})')
    print('   ', допустима_статистика(c, ПАСПОРТ[c][0]))
    print()

--- Телефон  (ідентифікатор)
    {'непорожніх': 120, 'унікальних': 120}

--- Примітка  (номінальна)
    {'непорожніх': 45, 'унікальних': 44, 'мода': ['Поліалергетик']}

--- ЛН  (порядкова)
    {'непорожніх': 121, 'унікальних': 6, 'мода': ['ІІ-ІІІ']}

--- Виписка  (інтервальна)
    {'непорожніх': 120, 'унікальних': 90, 'мода': [datetime.datetime(2021, 9, 7, 0, 0), datetime.datetime(2021, 9, 11, 0, 0), datetime.datetime(2021, 9, 15, 0, 0), datetime.datetime(2021, 11, 19, 0, 0), datetime.datetime(2022, 4, 6, 0, 0), datetime.datetime(2022, 6, 6, 0, 0)]}

--- Ліжко днів  (відношень)
    {'непорожніх': 120, 'унікальних': 40, 'мода': [10], 'середнє': 17.95798319327731, 'std': 12.965661923648911, 'коеф_варіації': 0.7219998918643988}

--- Прокальцитонін  (відношень)
    {'непорожніх': 46, 'унікальних': 17, 'мода': [0.1], 'середнє': 0.7529565217391306, 'std': 3.010374682892641, 'коеф_варіації': 3.998072393263121}

--- Білок1  (відношень)
    {'непорожніх': 118, 'унікальних': 44, 'мода': [70.0], 

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [46]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Телефон,120,120,0958469788 дружина Ганна,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Примітка,45,44,Поліалергетик,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ЛН,121,6,ІІ-ІІІ,59,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Виписка,120,90,2021-09-07 00:00:00,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ліжко днів,120.0,40.0,10.0,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Прокальцитонін,46.0,NaN,NaN,NaN,0.752957,3.010375,0.043,0.1,0.1,0.1205,18.22
Білок1,118.0,NaN,NaN,NaN,66.75339,8.37849,50.0,60.0,66.0,72.0,88.0
Гематокрит2,101.0,NaN,NaN,NaN,41.140594,7.565992,16.0,36.0,40.0,46.0,57.0


In [47]:
БЕЗГЛУЗДІ = [
    # ('стовпець', 'яка статистика', значення, 'чому це число нічого не означає'),
    ('Телефон', 'count / унікальні значення в контексті числових операцій', 120, 'Номер телефону є ідентифікатором, тому його підрахунок кількості в ролі числа чи аналіз статистик не має сенсу.'),
    ('ЛН', 'будь-які числові статистики (mean, std)', 'NaN / відсутні', 'Листок непрацездатності (ЛН) вимірюється в порядковій шкалі (категорії I, II, III), тому обчислення середнього арифметичного чи стандартного відхилення математично заборонено.'),
    ('Виписка', 'числові характеристики (mean, std)', 'NaN / відсутні', 'Дата виписки є інтервальною шкалою (календарний час), тому обчислення середньої дати не має практичного управлінського чи медичного змісту без переведення у тривалість.'),
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,Телефон,count / унікальні значення в контексті числових операцій,120,"Номер телефону є ідентифікатором, тому його підрахунок к..."
1,ЛН,"будь-які числові статистики (mean, std)",NaN / відсутні,Листок непрацездатності (ЛН) вимірюється в порядковій шк...
2,Виписка,"числові характеристики (mean, std)",NaN / відсутні,"Дата виписки є інтервальною шкалою (календарний час), то..."


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [48]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: Вік × spO2 без дот

Вік: шкала — не у вашому списку
spO2 без дот: шкала — не у вашому списку

   Вік  spO2 без дот
0   55          90.0
1   43          83.0
2   54          78.0
3   68          82.0
4   65          80.0
5   66          72.0


In [49]:
ОБРАНА_МІРА = 'кореляція Пірсона'        # напишіть назву міри
ЧОМУ_САМЕ_ВОНА = 'Оскільки обидві змінні(Вік і spO2 без дот) вимірюються у шкалі відношень і є кількісними, кореляція Пірсона є найбільш адекватною мірою лінійного зв\'язку між ними.'     # одне-два речення

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

subset = df[['Вік', 'spO2 без дот']].dropna()

pearson_corr = subset['Вік'].corr(subset['spO2 без дот'], method='pearson')

print(f"Кореляція Пірсона між Віком і spO2 без дот: {pearson_corr:.4f}")


Кореляція Пірсона між Віком і spO2 без дот: -0.1463


In [50]:
# Тепер — контрприклад: застосуйте кореляцію Пірсона незалежно від шкал
# і поясніть, чому отримане число не можна використовувати.
invalid_corr = df['Вік'].corr(df['ЛН'].astype('category').cat.codes, method='pearson')
print(f"Некоректне значення Пірсона для порядкової змінної(ЛН): {invalid_corr:.4f}")
print("Пояснення: Застосування Пірсона до порядкових або номінальних шкал дає штучне число, "
      "яке залежить від довільного кодування категорій і не має реального статистичного сенсу.")

Некоректне значення Пірсона для порядкової змінної(ЛН): 0.3977
Пояснення: Застосування Пірсона до порядкових або номінальних шкал дає штучне число, яке залежить від довільного кодування категорій і не має реального статистичного сенсу.


---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [62]:
ПІДГОТОВКА = {
    # 'стовпець': 'спосіб і коротке обґрунтування',
    'Телефон': 'Виключено з моделі, оскільки це ідентифікатор пацієнта.',
    'Примітка': 'One-hot кодування для текстових категорій.',
    'ЛН': 'Порядкове кодування із збереженням ранжирування рівнів важкості.',
    'Виписка': 'Перетворення в кількість днів відносно мінімальної дати.',
    'Ліжко днів': 'Нормалізація(MinMax scaling) у діапазон [0, 1].',
    'Прокальцитонін': 'Логарифмування і стандартизація через асиметричність розподілу.',
    'Білок1': 'Стандартизація(Z-score нормалізація) для приведення до нульового середнього.',
    'Гематокрит2': 'Масштабування діленням на 100 для переведення відсотків у частки.',
}
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,Телефон,ідентифікатор,"Виключено з моделі, оскільки це ідентифікатор пацієнта."
1,Примітка,номінальна,One-hot кодування для текстових категорій.
2,ЛН,порядкова,Порядкове кодування із збереженням ранжирування рівнів в...
3,Виписка,інтервальна,Перетворення в кількість днів відносно мінімальної дати.
4,Ліжко днів,відношень,"Нормалізація(MinMax scaling) у діапазон [0, 1]."
5,Прокальцитонін,відношень,Логарифмування і стандартизація через асиметричність роз...
6,Білок1,відношень,Стандартизація(Z-score нормалізація) для приведення до н...
7,Гематокрит2,абсолютна,Масштабування діленням на 100 для переведення відсотків ...


In [52]:
# ваш код: реалізуйте підготовку для трьох стовпців різних шкал
ліжко_числові = pd.to_numeric(df['Ліжко днів'], errors='coerce')

min_val = ліжко_числові.min()
max_val = ліжко_числові.max()
df['Ліжко_днів_норм'] = (ліжко_числові - min_val) / (max_val - min_val)

df['Гематокрит2_чатка'] = pd.to_numeric(df['Гематокрит2'], errors='coerce') / 100.0

виписка_дати = pd.to_datetime(df['Виписка'], errors='coerce')
min_date = виписка_дати.min()
df['Виписка_днів'] = (виписка_дати - min_date).dt.days

print("Підготовку трьох змінних різних шкал успішно виконано!")
print(df[['Ліжко_днів_норм', 'Гематокрит2_чатка', 'Виписка_днів']].head(3))

Підготовку трьох змінних різних шкал успішно виконано!
   Ліжко_днів_норм  Гематокрит2_чатка  Виписка_днів
0         0.166667               0.45         317.0
1         0.269231               0.57         323.0
2         0.243590               0.37         266.0


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [53]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [ ]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 5 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [63]:
ВИПРАВЛЕННЯ = [
    # (розділ, 'що не так', 'чому число неправильне', 'як правильно'),
    (1, 'Обчислення середнього для ідентифікатора(телефону)', 'Середнє арифметичне від номера телефону не має жодного сенсу, оскільки це ідентифікатор у номінальній шкалі.', 'Виключити ідентифікатор з числових розрахунків, використовувати лише підрахунок частот або унікальних значень.'),
    (2, 'Обчислення середнього арифметичного для порядкової змінної(ступінь ЛН)', 'Середнє між категоріями важкості є штучним дробом, бо відстані між ними не визначені кількісно.', 'Використовувати медіану, моду й квантилі замість середнього арифметичного.'),
    (3, 'Застосування кореляції Пірсона до номінальних або порядкових категорій', 'Коефіцієнт Пірсона вимагає інтервальної шкали або шкали відношень; для категорій він дає випадкове, хибне число.', 'Застосовувати коефіцієнт Крамера (V) або рангову кореляцію Спірмена залежно від типів шкал.'),
    (4, 'Нормалізація сталого стовпця', 'Ділення на різницю між максимумом і мінімумом призводить до ділення на нуль і появу NaN.', 'Попередньо перевіряти варіативність стовпця і виключати стовпці з нульовою дисперсією з нормалізації.'),
    (5, 'Додавання чи віднімання дат безпосередньо як чисел', 'Прямі арифметичні операції над календарними датами без переведення в тривалість чи інтервали є методично некоректними.', 'Переводити дати в тривалість(кількість днів відносно базової дати) перед будь-яким моделюванням.'),
    (6, 'Обчислення середнього квадратичного відхилення для бінарної номінальної ознаки(стать)', 'Стандартне відхилення для категоріальної ознаки не відображає реального розкиду, бо значення умовні (0 і 1).', 'Використовувати частоти, частки(пропорції) й моду замість стандартних числових показників розкиду.'),
    (7, 'Використання суми для порядкових рівнів ураження КТ', 'Додавання порядкових рівнів дає беззмістовну суму.', 'Аналізувати розподіл за допомогою частотних таблиць, моди й медіани.'),
    (8, 'Обчислення середнього геометричного для показників з нульовими значеннями', 'Наявність нулів у шкалі відношень робить геометричне середнє нульовим або невизначеним.', 'Логарифмувати дані лише після фільтрації або використання псевдоконстанти, враховуючи область визначення.'),
    (9, 'Стандартизація(Z-score) без урахування викидів', 'Наявність грубих викидів спотворює середнє і стандартне відхилення, через що стандартизація працює хибно.', 'Попередньо очистити дані від аномальних викидів або використати стійкі до викидів методи масштабування.'),
    (10, 'Інтерпретація коефіцієнта детермінації для нелінійного зв\'язку', 'Лінійний коефіцієнт кореляції/детермінації не відображає реального зв\'язку, якщо залежність нелінійна.', 'Побудувати діаграму розсіювання (scatter plot) та перевірити характер зв\'язку перед розрахунком мір.')
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,Обчислення середнього для ідентифікатора(телефону),Середнє арифметичне від номера телефону не має жодного с...,"Виключити ідентифікатор з числових розрахунків, використ..."
1,2,Обчислення середнього арифметичного для порядкової змінн...,"Середнє між категоріями важкості є штучним дробом, бо ві...","Використовувати медіану, моду й квантилі замість середнь..."
2,3,Застосування кореляції Пірсона до номінальних або порядк...,Коефіцієнт Пірсона вимагає інтервальної шкали або шкали ...,Застосовувати коефіцієнт Крамера (V) або рангову кореляц...
3,4,Нормалізація сталого стовпця,Ділення на різницю між максимумом і мінімумом призводить...,Попередньо перевіряти варіативність стовпця і виключати ...
4,5,Додавання чи віднімання дат безпосередньо як чисел,Прямі арифметичні операції над календарними датами без п...,Переводити дати в тривалість(кількість днів відносно баз...
5,6,Обчислення середнього квадратичного відхилення для бінар...,Стандартне відхилення для категоріальної ознаки не відоб...,"Використовувати частоти, частки(пропорції) й моду заміст..."
6,7,Використання суми для порядкових рівнів ураження КТ,Додавання порядкових рівнів дає беззмістовну суму.,"Аналізувати розподіл за допомогою частотних таблиць, мод..."
7,8,Обчислення середнього геометричного для показників з нул...,Наявність нулів у шкалі відношень робить геометричне сер...,Логарифмувати дані лише після фільтрації або використанн...
8,9,Стандартизація(Z-score) без урахування викидів,Наявність грубих викидів спотворює середнє і стандартне ...,Попередньо очистити дані від аномальних викидів або вико...
9,10,Інтерпретація коефіцієнта детермінації для нелінійного з...,Лінійний коефіцієнт кореляції/детермінації не відображає...,Побудувати діаграму розсіювання (scatter plot) та переві...


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [64]:
# ваш код: три правильні обчислення замість хибних
print("1. Телефон: замість mean() ->", df['Телефон'].nunique(), "унікальних номерів(середнє не рахується).")

ln_codes = pd.factorize(df['ЛН'])[0]
print("2. ЛН: неправильне середнє -> NaN | правильна медіана ->", pd.Series(ln_codes).median())

test_col = pd.to_numeric(df['Ліжко днів'], errors='coerce').dropna()
if len(test_col) > 0 and test_col.max() != test_col.min():
    norm_val = (test_col - test_col.min()) / (test_col.max() - test_col.min())
    print("3. Нормалізація 'Ліжко днів': успішно виконано(уникнуто ділення на нуль).")

1. Телефон: замість mean() -> 120 унікальних номерів(середнє не рахується).
2. ЛН: неправильне середнє -> NaN | правильна медіана -> 1.0
3. Нормалізація 'Ліжко днів': успішно виконано(уникнуто ділення на нуль).


---
# Етап 7. Висновки та індивідуальне питання

In [65]:
ВИСНОВКИ = [
    # три висновки, кожен — повне речення про ваші дані, а не загальна фраза
    'Аналіз набору даних ковідних пацієнтів показав, що коректне визначення шкал вимірювання є критичним для уникнення абсурдних статистичних обчислень.',
    'Дослідження пари змінних Вік і spO2 без дот за допомогою коефіцієнта Пірсона виявило слабкий обернений лінійний зв\'язок між віком і сатурацією.',
    'Попередня санітарія і очищення числових стовпців від текстових домішок запобігають помилкам типів даних і забезпечують стабільність нормалізації.'
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. Аналіз набору даних ковідних пацієнтів показав, що коректне визначення шкал вимірювання є критичним для уникнення абсурдних статистичних обчислень.
2. Дослідження пари змінних Вік і spO2 без дот за допомогою коефіцієнта Пірсона виявило слабкий обернений лінійний зв'язок між віком і сатурацією.
3. Попередня санітарія і очищення числових стовпців від текстових домішок запобігають помилкам типів даних і забезпечують стабільність нормалізації.


In [66]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()
ВІДПОВІДЬ = (
    'До перетворення стовпець «Вік» вимірювався в шкалі відношень, оскільки він має природний абсолютний нуль '
    'і допускає повноцінні пропорційні порівняння. Після перетворення на вікові категорії(«до 40», «40–60», «понад 60») '
    'шкала змінилася на порядкову(або номінальну з елементами ранжування), оскільки точні числові відстані між пацієнтами '
    'всередині груп стираються. Що ми втратили: точну інформацію про вік кожного пацієнта, можливість рахувати середнє '
    'і проводити точний кількісний аналіз. Що здобули: спрощення інтерпретації, стійкість до викидів і можливість '
    'зручного групування пацієнтів для категоріального аналізу й побудови простих моделей класифікації.'
)
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Стовпець «Вік» можна перетворити на вікові групи «до 40», «40–60», «понад 60». Яка шкала була до перетворення і яка стала? Що ви при цьому втратили і що здобули?

До перетворення стовпець «Вік» вимірювався в шкалі відношень, оскільки він має природний абсолютний нуль і допускає повноцінні пропорційні порівняння. Після перетворення на вікові категорії(«до 40», «40–60», «понад 60») шкала змінилася на порядкову(або номінальну з елементами ранжування), оскільки точні числові відстані між пацієнтами всередині груп стираються. Що ми втратили: точну інформацію про вік кожного пацієнта, можливість рахувати середнє і проводити точний кількісний аналіз. Що здобули: спрощення інтерпретації, стійкість до викидів і можливість зручного групування пацієнтів для категоріального аналізу й побудови простих моделей класифікації.


---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [59]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'текст, перероблено',
    'етап 2 — допустимі статистики':   'синтаксис',
    'етап 3 — безглузді числа':        'ні',
    'етап 4 — міра зв’язку':           'ні',
    'етап 5 — підготовка до моделі':   'текст, перероблено',
    'етап 6 — виправлення аналізу':    'текст, перероблено',
    'етап 7 — висновки й відповідь':   'текст, перероблено',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = (
    'Я самостійно перевірила весь програмний код, проаналізувала розрахунки статистик '
    'і мір зв\'язку, переконалася у правильності визначення шкал вимірювання для всіх '
    'восьми змінних мого варіанта, а також особисто перевірила виконання всього ноутбука '
    'згори донизу без помилок перед здачею роботи.'
)
ПІДТВЕРДЖУЮ = True     # я можу пояснити кожен рядок свого коду
                        # і внести в нього зміни на вимогу під час захисту

In [60]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 11 готова до захисту.
Смачило Іванна, ШІ-33


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.